# One-cell VS Code on Kaggle

**Setup (once):** Settings -> enable **Internet**, and Add-ons -> Secrets -> add `NGROK_TOKEN`.

**Every session:** just run the single cell below, then open the printed URL.
In the VS Code terminal, type `klogin` to sign in to Kiro CLI.

In [ ]:
import os, subprocess, time
from pathlib import Path

PORT = 10000
REPO = "https://github.com/Hvkki/Sito-per-intelligenti.git"
WORKDIR = "/kaggle/working"

# 1) ngrok token from Kaggle Secrets
try:
    from kaggle_secrets import UserSecretsClient
    os.environ["NGROK_TOKEN"] = UserSecretsClient().get_secret("NGROK_TOKEN")
except Exception:
    pass
TOKEN = os.environ.get("NGROK_TOKEN", "")
assert TOKEN, "Add a Kaggle Secret named NGROK_TOKEN (Add-ons -> Secrets) and attach it."

# 2) Install tools only if they are missing
subprocess.run("command -v code-server >/dev/null 2>&1 || curl -fsSL https://code-server.dev/install.sh | sh", shell=True)
subprocess.run("test -x $HOME/.local/bin/kiro-cli || curl -fsSL https://cli.kiro.dev/install | bash", shell=True)
subprocess.run("pip install -q pyngrok", shell=True, check=True)

# 3) VS Code terminal convenience: PATH + a 'klogin' shortcut for Kiro CLI sign-in
bashrc = Path.home() / ".bashrc"
snippet = (
    "\n# --- kaggle code-server helpers ---\n"
    'export PATH="$HOME/.local/bin:$PATH"\n'
    "alias klogin='kiro-cli login --license pro "
    "--identity-provider https://d-906673ba2c.awsapps.com/start "
    "--region us-east-1 --use-device-flow'\n"
)
txt = bashrc.read_text() if bashrc.exists() else ""
if "klogin" not in txt:
    bashrc.write_text(txt + snippet)

# 4) Clone your repo so your work persists (commit + push to save)
dest = os.path.join(WORKDIR, "Sito-per-intelligenti")
if not os.path.exists(dest):
    subprocess.run(f"git clone {REPO} {dest}", shell=True)

# 5) Start the tunnel + code-server (opens your repo folder)
from pyngrok import ngrok
ngrok.set_auth_token(TOKEN)
ngrok.kill()
url = ngrok.connect(PORT, "http").public_url
subprocess.Popen(
    ["code-server", "--bind-addr", f"0.0.0.0:{PORT}", "--auth", "none", dest],
    env=os.environ.copy(),
)
time.sleep(5)

print("=" * 60)
print(" OPEN VS CODE :", url)
print(" Your files   :", dest, "(git push to save your work)")
print(" Kiro CLI     : in the VS Code terminal, type:  klogin")
print("=" * 60)